# Parameterized request to a local model through ZEMI Arsenal

This example selects one model by its component playbook parameter, starts only that model, and sends a short message.

## Input parameters

In [ ]:
arsenal_config_path = "@comp/zemi/llm_curated_set_model_mode.toml"
arsenal_stop_before_playbook_begin = True
arsenal_stop_after_playbook_end = True
model_name = "lfm2_350m"

## Playbook preparation

In [ ]:
# Automatically reload imported modules when their source code changes
%load_ext autoreload
%autoreload 2

# Set the working directory to the ZEMI component root
from pathlib import Path

while not Path(".zemicomp").is_file():
    if Path.cwd().parent == Path.cwd():
        raise FileNotFoundError("Could not find the ZEMI component root")
    %cd ..
PROJECT_ROOT = Path.cwd()
PROJECT_ROOT

In [ ]:
from zemi.arsenal.python import PythonVenv

# Verify the base Python venv, Z-bundle, and active C-bundle.
PythonVenv.from_config("@comp/00_init.toml").verify()

## Starting Arsenal in Model Mode

In [ ]:
import zemi
from zemi.arsenal import ArsenalSession

arsenal = ArsenalSession(arsenal_config_path)
zemi.arsenal.begin(
    arsenal,
    stop_before_begin=arsenal_stop_before_playbook_begin,
)

## Model request

In [ ]:
model = arsenal.model(model_name)
assistant = model.assistants["assistant"]
client = assistant.clients.openai.client

In [ ]:
response = client.chat.completions.create(
    model=assistant.clients.model,
    messages=[{"role": "user", "content": "Hello!"}],
)
print(response.choices[0].message.content)

## Stopping Arsenal

After the request, end the session and stop the local server.

In [ ]:
zemi.arsenal.end(
    arsenal,
    stop_after_end=arsenal_stop_after_playbook_end,
)

## Output parameters

Publish structured JSON results for the ZEMI job report. The custom MIME output is the marker; no cell tag is required.

In [ ]:
from zemi.playbook import output_params

model_response = response.choices[0].message.content
output_params({
    "model_response": model_response,
    "response_length": len(model_response),
})